# The agent as an experimental subject — a causal study

A RAG agent makes a **choice**: when the retrieved context looks thin, it decides to *re-query*.
This notebook measures the causal effect **of that choice** — not on fixed inputs, but on the
model's own endogenous behavior. That's model/behavior evaluation, the part teams struggle with
once they ship agents.

We assign the *option* to re-query; the model chooses *uptake*. Two estimands fall out:
- **ITT** — the effect of *offering* self-correction, averaged over **all** questions. The ship-it number.
- **CACE** — the effect among the **compliers**: questions where the agent actually re-queried.

They're linked by `ITT = P(re-query) × CACE`. Because the subject is a model (no memory across API
calls), we run the *same* question through both arms — a **crossover** — and read the complier
effect off directly, no instrumental variables needed.

---
### How this notebook is organized
- **Part 1 — the study, as code.** The full data-generating pipeline (agent, judge, experiment).
  It was run **once** on a Colab GPU+API session; the raw results are saved in `study.json`. The
  live experiment call is commented out — **Part 1 is reference, not meant to be re-run.**
- **Part 2 — the analysis.** Loads `study.json` and reproduces every estimand, test, and figure.
  No API calls; runs in seconds. **Start here if you just want the result.**

## Part 1 — The study, as code  *(reference — run once to generate `study.json`)*

> These cells make live SEC/network requests and Claude API calls. They're included as the record
> of how the data was produced; you don't need to run them — Part 2 works entirely from the saved file.

In [ ]:
!pip install -qU langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -qU qdrant-client sentence-transformers beautifulsoup4 lxml anthropic

In [ ]:
import os, re, time, textwrap
import requests
import numpy as np
from bs4 import BeautifulSoup
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

import anthropic
from pydantic import BaseModel
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from sentence_transformers import CrossEncoder

SEC_USER_AGENT = "David Schaaf davidschaaf@berkeley.edu"   # edit to your name/email
AGENT_MODEL = "claude-opus-4-8"
MAX_STEPS = 5
client = anthropic.Anthropic()
print("Setup ready.")

### Retrieval + the agent's tool
Fetch → clean → chunk (each chunk keeps a stable `chunk_id`) → embed → index → reranked retrieval,
exposed to the agent as a `search_filings` tool.

In [ ]:
COMPANIES = {"AAPL": 320193, "MSFT": 789019, "NVDA": 1045810}

def fetch_latest_10k_html(ticker, cik):
    h = {"User-Agent": SEC_USER_AGENT}
    recent = requests.get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json",
                          headers=h).json()["filings"]["recent"]
    i = next(j for j, f in enumerate(recent["form"]) if f == "10-K")
    acc, doc = recent["accessionNumber"][i].replace("-", ""), recent["primaryDocument"][i]
    html = requests.get(f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/{doc}",
                        headers=h).text
    time.sleep(0.5)
    return html

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
documents = []
for tk, cik in COMPANIES.items():
    print("Fetching", tk, "...")
    soup = BeautifulSoup(fetch_latest_10k_html(tk, cik), "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    for chunk in splitter.split_text(re.sub(r"\s+", " ", soup.get_text(" ")).strip()):
        documents.append(Document(page_content=chunk,
            metadata={"chunk_id": len(documents), "ticker": tk}))

embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")
qdrant = QdrantVectorStore.from_documents(
    documents, embeddings, location=":memory:", collection_name="sec_causal")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_reranked(query, fetch_k=10, top_k=4):
    cands = qdrant.similarity_search(query, k=fetch_k)
    scores = reranker.predict([(query, d.page_content) for d in cands])
    return [d for _, d in sorted(zip(scores, cands), key=lambda x: x[0], reverse=True)][:top_k]

def do_search(query, k=4):
    docs = retrieve_reranked(query, top_k=k)
    ids = [d.metadata["chunk_id"] for d in docs]
    text = "\n\n".join(f"[id={d.metadata['chunk_id']} | {d.metadata['ticker']}] {d.page_content}"
                       for d in docs)
    return text, ids

SEARCH_TOOL = {
    "name": "search_filings",
    "description": ("Search the SEC 10-K filings for passages relevant to a query. If the results "
                    "don't fully answer the question, search again with a refined query."),
    "input_schema": {"type": "object",
                     "properties": {"query": {"type": "string", "description": "What to search for"}},
                     "required": ["query"]},
}
SYSTEM = ("You answer questions about SEC 10-K filings for Apple, Microsoft, and NVIDIA using the "
          "search_filings tool. If the passages don't fully answer the question, refine your query "
          "and search again. Answer only from retrieved passages, cite ids like [42], and if the "
          "filings don't contain the answer, say so. Be concise.")
print(f"Retrieval + tool ready over {len(documents)} chunks.")

### The capped agent — the one-knob manipulation
One agent function with a `max_hops` cap. **Control** = `max_hops=1` (one search, then forced to
answer); **treatment** = `max_hops=MAX_STEPS` (free to re-query). Both share the first hop, so the
only difference is the extra retrieval.

In [ ]:
def run_agent_capped(question, max_hops):
    question_prompt = {"role": "user", "content": question}
    messages = [question_prompt]
    searches = []
    tokens = 0

    # SEARCH_TOOL available
    for _ in range(MAX_STEPS):
        kw = dict(model=AGENT_MODEL, max_tokens=1024, system=SYSTEM,
                  messages=messages)
        if len(searches) < max_hops:
            kw["tools"] = [SEARCH_TOOL]

        response = client.messages.create(**kw)
        tokens += response.usage.input_tokens + response.usage.output_tokens
        if response.stop_reason != "tool_use":
            break

        # Only if we need to use the tool search_filings
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                query = block.input.get("query", "")
                searches.append(query)
                text, _ = do_search(**block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": text})
        messages.append({"role": "user", "content": tool_results})

    if response.stop_reason == "tool_use":
        kw = dict(model=AGENT_MODEL, max_tokens=1024, system=SYSTEM,
                  messages=messages)
        response = client.messages.create(**kw)
        tokens += response.usage.input_tokens + response.usage.output_tokens

    answer = "".join(b.text for b in response.content if b.type == "text")
    return {"answer": answer, "hops": len(searches), "tokens": tokens}

print("Agent ready.")

### The outcome — a blind completeness judge
Re-querying should improve **completeness** (a one-hop answer can be faithful to thin context yet
miss half the question). A Claude judge scores each answer 1–5; it never sees which arm produced
the answer, so it's **blind by construction**.

In [ ]:
class Completeness(BaseModel):
    score: int   # 1 (misses most of the question) .. 5 (fully addresses every part)
    reason: str

def judge_completeness(question, answer):
    prompt = ("Score how COMPLETELY the answer addresses the question, 1-5 "
              "(1=misses most, 5=addresses every part). Judge completeness only.\n\n"
              f"Question: {question}\n\nAnswer: {answer}")
    return client.messages.parse(model=AGENT_MODEL, max_tokens=300,
        messages=[{"role": "user", "content": prompt}], output_format=Completeness).parsed_output

print("Blind completeness judge ready.")

### The question set
Weighted toward **compound / multi-part** questions (which give the agent a reason to re-query),
with three simpler ones so never-takers are represented.

In [ ]:
QUESTIONS = [
    "What are NVIDIA's main risk factors, and how does it describe competition?",
    "What does Apple say about its revenue, and what risks does it tie to supply chain?",
    "How does Microsoft describe its cloud business and the competition it faces there?",
    "What does NVIDIA say about data-center demand and the risks to that demand?",
    "Summarize Apple's regulatory risks and its risks related to foreign operations.",
    "How does Microsoft describe both its AI investments and the risks around them?",
    "What are Apple's main product risks, and what does it say about R&D spending?",
    "What does NVIDIA say about its supply concentration and about intellectual property?",
    "How does Microsoft describe its competition, and what does it say about cybersecurity risk?",
    "What are the main risk factors Apple highlights?",            # simpler
    "How does NVIDIA describe its gaming segment?",                # simpler
    "What does Microsoft say about its revenue sources?",          # simpler
]
N_QUESTIONS = len(QUESTIONS)
R = 3   # independent runs per question per arm
print(f"{N_QUESTIONS} questions x {R} runs x 2 arms = {N_QUESTIONS*R*2} agent trajectories.")

### The crossover experiment
For each question, `R` independent replicates; each replicate runs **both** arms and judges both
answers. Logged per run: `Y_control`, `Y_treated`, `D` (did treatment re-query — `hops >= 2`), tokens.

> The live run below is **commented out** — it's what produced `study.json` (~$5–8 in API calls).
> Part 2 loads that file instead.

In [ ]:
def run_experiment(questions, repeats):
    output = []
    for question_idx, question in enumerate(questions):
        for r in range(repeats):
            ctrl = run_agent_capped(question, 1)
            trt = run_agent_capped(question, MAX_STEPS)
            output.append({
                "q_idx": question_idx,
                "run": r,
                "Y_control": judge_completeness(question, ctrl["answer"]).score,
                "Y_treated": judge_completeness(question, trt["answer"]).score,
                "D": int(trt["hops"] >= 2),
                "hops": trt["hops"],
                "tokens": ctrl["tokens"] + trt["tokens"]
            })
        print(f"  done q{question_idx+1}/{len(questions)}")
    return output

# --- LIVE RUN (commented out — this is what generated study.json) ---
# records = run_experiment(QUESTIONS, R)
# print(f"logged {len(records)} runs")

## Part 2 — Analysis  *(reproduces every result & figure from `study.json`, no API calls)*

Start here. This half is self-contained: it needs only `numpy` and `matplotlib`, plus `study.json`
in the working directory (in Colab, upload it or clone the repo).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

study = json.load(open("study.json"))
records   = study["records"]
QUESTIONS = study["questions"]
N_QUESTIONS = len(QUESTIONS)
MAX_STEPS = 5                      # the treatment cap used to generate the data
print(f"Loaded {len(records)} runs across {N_QUESTIONS} questions from study.json")

In [ ]:
# peek at the raw log
print(f"{'q':>3}{'run':>4}{'Yc':>4}{'Yt':>4}{'D':>3}{'hops':>5}")
for rec in records[:8]:
    print(f"{rec['q_idx']:>3}{rec['run']:>4}{rec['Y_control']:>4}{rec['Y_treated']:>4}"
          f"{rec['D']:>3}{rec['hops']:>5}")
print(f"\noverall re-query rate: {np.mean([r['D'] for r in records]):.2f}")

### Estimands — ITT, P(re-query), CACE
Point estimates use every run. CACE differences each complier against **its own** control run
(the crossover), so the within-run comparison is unconfounded — no IV needed.

In [ ]:
diffs = np.array([r["Y_treated"] - r["Y_control"] for r in records], float)
D = np.array([r["D"] for r in records])

ITT = np.mean(diffs)
p_requery = np.mean(D)
CACE = diffs[D == 1].mean() if D.any() else float("nan")

print(f"ITT (ship-it effect):        {ITT:+.3f}")
print(f"P(re-query):                 {p_requery:.3f}")
print(f"CACE (effect on compliers):  {CACE:+.3f}")
print(f"check  P(re-query) x CACE =  {p_requery * CACE:+.3f}   (should ~= ITT)")

### Inference — respect the clustering
The `R` runs of one question are repeated measures, not independent observations — the unit of
inference is the **question**. ITT gets a **sign-flip permutation test** over per-question effects;
the CIs come from a **cluster bootstrap** that resamples questions (each brings all its runs).

In [ ]:
rng = np.random.default_rng(0)
by_q = {q_idx: [r for r in records if r["q_idx"] == q_idx] for q_idx in range(N_QUESTIONS)}

# a - randomization inference
d_q = np.array([np.mean([r["Y_treated"] - r["Y_control"] for r in recs]) for recs in by_q.values()])
obs = np.mean(d_q)
null = np.array([(rng.choice([-1, 1], size=len(d_q)) * d_q).mean() for _ in range(5000)])
p_itt = np.mean(np.abs(null) >= abs(obs))

# b - clustering
def cluster_bootstrap_ci(estimand_fn, n_boot=3000):
    question_clusters = list(by_q.values())        # each cluster = one question's list of runs
    n_questions = len(question_clusters)
    bootstrap_estimates = []

    for _ in range(n_boot):
        # resample QUESTIONS (clusters) with replacement, then pull in all of their runs
        sampled_question_indices = rng.choice(n_questions, size=n_questions)
        resampled_runs = [
            run
            for question_index in sampled_question_indices
            for run in question_clusters[question_index]
        ]

        estimate = estimand_fn(resampled_runs)
        if not np.isnan(estimate):                 # skip resamples where the estimand is undefined
            bootstrap_estimates.append(estimate)

    return np.percentile(bootstrap_estimates, [2.5, 97.5])


def itt_estimand(runs):
    return np.mean([run["Y_treated"] - run["Y_control"] for run in runs])


def cace_estimand(runs):
    complier_runs = [run for run in runs if run["D"] == 1]
    if not complier_runs:                          # no compliers in this resample -> undefined
        return float("nan")
    return np.mean([run["Y_treated"] - run["Y_control"] for run in complier_runs])

itt_ci = cluster_bootstrap_ci(itt_estimand)
cace_ci = cluster_bootstrap_ci(cace_estimand)

print(f"ITT  {ITT:+.3f}   95% CI [{itt_ci[0]:+.3f}, {itt_ci[1]:+.3f}]   perm p = {p_itt:.3f}")
print(f"CACE {CACE:+.3f}   95% CI [{cace_ci[0]:+.3f}, {cace_ci[1]:+.3f}]")

### Figures

In [ ]:
from matplotlib.patches import Patch

# Okabe-Ito colorblind-safe palette (canonical CVD-safe set)
BLUE, ORANGE, VERM, GRAY = "#0072B2", "#E69F00", "#D55E00", "#7A7A7A"
INK, GRID = "#2b2b2b", "#dddddd"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.edgecolor": "#999999",
                     "axes.linewidth": 0.8, "text.color": INK, "axes.labelcolor": INK,
                     "xtick.color": INK, "ytick.color": INK})
def _clean(ax, left=True):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    if not left: ax.spines["left"].set_visible(False)
    ax.grid(color=GRID, linewidth=0.7); ax.set_axisbelow(True)

hops = np.array([r["hops"] for r in records])

# Fig 1 - ITT vs CACE with 95% CI
fig, ax = plt.subplots(figsize=(6.2, 2.9))
ests, y = [ITT, CACE], [1, 0]
xerr = [[ITT - itt_ci[0], CACE - cace_ci[0]], [itt_ci[1] - ITT, cace_ci[1] - CACE]]
ax.axvline(0, color=GRAY, ls="--", lw=1, zorder=1)
ax.errorbar(ests, y, xerr=xerr, fmt="o", color=BLUE, ecolor=BLUE, elinewidth=2.2,
            capsize=4, markersize=10, zorder=3)
for e, yy in zip(ests, y):
    ax.annotate(f"{e:+.2f}", (e, yy), textcoords="offset points", xytext=(0, 11),
                ha="center", fontweight="bold")
ax.set_yticks(y); ax.set_yticklabels(["ITT\n(all questions)", "CACE\n(compliers)"])
ax.set_ylim(-0.6, 1.6); ax.set_xlim(-0.4, 3.4)
ax.set_xlabel("effect on completeness (1-5 scale)")
ax.set_title("Re-querying's effect: ITT vs CACE (95% CI)", loc="left", fontweight="bold")
_clean(ax, left=False); ax.text(0.02, -0.5, "0 = no effect", color=GRAY, fontsize=9)
plt.tight_layout(); plt.show()

# Fig 2 - per-question heterogeneity
qids = sorted(by_q)
mean_eff = np.array([np.mean([r["Y_treated"] - r["Y_control"] for r in by_q[q]]) for q in qids])
is_simple = np.array([q >= 9 for q in qids])   # last 3 are the simpler questions
order = np.argsort(mean_eff)
fig, ax = plt.subplots(figsize=(6.6, 4))
colors = [ORANGE if is_simple[i] else BLUE for i in order]
ax.barh(range(len(order)), mean_eff[order], color=colors, height=0.72, zorder=3)
ax.set_yticks(range(len(order))); ax.set_yticklabels([f"Q{qids[i]}" for i in order])
ax.set_xlabel("mean effect on completeness")
ax.set_title("Effect is heterogeneous - largest on compound questions", loc="left", fontweight="bold")
ax.legend(handles=[Patch(color=BLUE, label="compound"), Patch(color=ORANGE, label="simpler")],
          frameon=False, loc="lower right")
_clean(ax, left=False)
plt.tight_layout(); plt.show()

# Fig 3 - hops histogram with the parallel-tool tell
fig, ax = plt.subplots(figsize=(6.2, 3.2))
vals, counts = np.unique(hops, return_counts=True)
ax.bar(vals, counts, color=[VERM if v > MAX_STEPS else BLUE for v in vals], width=0.72, zorder=3)
ax.set_xlabel("searches per treatment run (hops)"); ax.set_ylabel("runs"); ax.set_xticks(vals)
ax.set_title("hops > MAX_STEPS (5) can only come from parallel tool calls", loc="left", fontweight="bold")
tell = np.where(vals > MAX_STEPS)[0]
if len(tell):
    v, c = vals[tell[0]], counts[tell[0]]
    ax.annotate("parallel\ntool use", (v, c), textcoords="offset points", xytext=(-4, 22),
                ha="center", color=VERM, fontsize=9, arrowprops=dict(arrowstyle="->", color=VERM))
_clean(ax)
plt.tight_layout(); plt.show()

## What the numbers say — and what they don't

At face value this is a home run: the multi-hop agent's answers score **+2.25** higher on
completeness (95% CI [1.4, 3.1], permutation *p* ≈ 0.003). But the honest read is more interesting
than the headline, and it's the whole point of doing the inference carefully:

- **The compliance design collapsed.** `P(re-query) = 0.97` — almost every run re-queried, ~1
  never-taker in 36. So ITT ≈ CACE *by construction* (`0.97 × 2.31 = 2.25` ✓), and the ITT-vs-CACE
  machinery told us nothing *on this data*. The tiny gap between them is that single never-taker
  diluting ITT — the mechanism, visible at n=1.
- **Compliance was contaminated by parallel tool use.** `hops` reaches **6** with a cap of 5
  (Figure 3) — impossible without the model firing multiple searches in one turn. `D = hops >= 2`
  therefore conflates "re-queried after reading" with "searched twice at once," pushing `P(re-query)`
  toward 1.
- **The outcome saturated.** `Y_treated` is 5 in 30/36 runs; the judge can't distinguish a great
  multi-hop answer from a merely good one, so the effect is really "how bad the one-shot control is."
- **But a real signal survives (Figure 2):** the effect is strongly heterogeneous — ~+4 on compound
  questions, ~0 on several. Iteration helps exactly where one retrieval is insufficient, and barely
  at all where it isn't. That's a genuine finding, not an artifact.

**The takeaway is the critical read, not the big number.** A v2 that (a) disables parallel tool use
so `hops` = re-query turns, (b) rebalances toward more single-fact questions to create real
never-takers (target `P(re-query)` ≈ 0.5), and (c) uses a finer outcome to break the ceiling, would
let the ITT/CACE distinction actually demonstrate. And in production — where you can't re-run the
same query under both arms — the crossover is gone and you'd recover CACE with instrumental
variables (`ITT_Y / ITT_D`); same estimand, harder identification.

*Design grounded in Bottou et al. (2013), "Counterfactual Reasoning and Learning Systems" (JMLR),
and Frauen et al. (2026), "Causal Methods for LLM Development and Evaluation"; ITT/CACE from
Angrist, Imbens & Rubin (1996), JASA.*